# Market Strategy Backtest

A focused market-pattern notebook. The indicator math is deterministic, but the trading edge is heuristic until tested with point-in-time data, fees, slippage, and walk-forward validation.

## Setup
JupyterLite cannot reliably install live market clients such as yfinance. This notebook uses the local OHLCV sample and focuses on backtest mechanics.

In [ ]:
%pip install pandas numpy matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = '/cases/datasets/market_ohlcv_sample.csv'
TRADING_DAYS = 252
COST_PER_TURNOVER = 0.0005

def read_portal_csv(path):
    site_path = path if path.startswith('/') else f'/{path}'
    try:
        open_url = __import__('pyodide.http', fromlist=['open_url']).open_url
        return pd.read_csv(open_url(site_path))
    except Exception:
        relative = site_path.lstrip('/')
        candidates = [Path(relative), Path('..') / relative, Path.cwd() / relative, Path.cwd().parent / relative]
        for candidate in candidates:
            if candidate.exists():
                return pd.read_csv(candidate)
        raise FileNotFoundError(f'Could not find {site_path}')

px = read_portal_csv(DATA_PATH)
px['date'] = pd.to_datetime(px['date'])
px = px.sort_values('date').set_index('date')
px['return'] = px['close'].pct_change().fillna(0)
print('Rows:', len(px))
display(px)

In [ ]:
def summarize_strategy(frame, return_col, turnover_col):
    returns = frame[return_col].fillna(0)
    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1
    volatility = returns.std()
    sharpe = np.nan if volatility == 0 else returns.mean() / volatility * np.sqrt(TRADING_DAYS)
    return {
        'total_return': round(float(equity.iloc[-1] - 1), 4),
        'annualized_sharpe': round(float(sharpe), 4) if not np.isnan(sharpe) else np.nan,
        'max_drawdown': round(float(drawdown.min()), 4),
        'annual_turnover': round(float(frame[turnover_col].mean() * TRADING_DAYS), 4),
    }

close = px['close']
backtest = px.copy()
backtest['fast_ma'] = close.rolling(3).mean()
backtest['slow_ma'] = close.rolling(5).mean()
backtest['raw_signal'] = (backtest['fast_ma'] > backtest['slow_ma']).astype(int)
backtest['position'] = backtest['raw_signal'].shift(1).fillna(0)
backtest['turnover'] = backtest['position'].diff().abs().fillna(backtest['position'].abs())
backtest['cost'] = backtest['turnover'] * COST_PER_TURNOVER
backtest['strategy_return'] = backtest['position'] * backtest['return'] - backtest['cost']
backtest['buy_hold_return'] = backtest['return']
backtest['strategy_equity'] = (1 + backtest['strategy_return'].fillna(0)).cumprod()
backtest['buy_hold_equity'] = (1 + backtest['buy_hold_return'].fillna(0)).cumprod()
backtest['strategy_drawdown'] = backtest['strategy_equity'] / backtest['strategy_equity'].cummax() - 1
backtest['buy_hold_drawdown'] = backtest['buy_hold_equity'] / backtest['buy_hold_equity'].cummax() - 1

strategy_summary = summarize_strategy(backtest, 'strategy_return', 'turnover')
buy_hold = backtest.assign(bh_turnover=0)
buy_hold_summary = summarize_strategy(buy_hold, 'buy_hold_return', 'bh_turnover')
summary = pd.DataFrame([strategy_summary, buy_hold_summary], index=['moving_average_strategy', 'buy_and_hold'])

display(backtest[['close', 'fast_ma', 'slow_ma', 'position', 'turnover', 'cost', 'strategy_equity', 'buy_hold_equity']])
display(summary)

plt.figure(figsize=(8, 4))
plt.plot(backtest.index, backtest['strategy_equity'], marker='o', label='strategy')
plt.plot(backtest.index, backtest['buy_hold_equity'], marker='o', label='buy and hold')
plt.title('Equity curve after costs')
plt.ylabel('Growth of 1.0')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(backtest.index, backtest['strategy_drawdown'], marker='o', label='strategy drawdown')
plt.plot(backtest.index, backtest['buy_hold_drawdown'], marker='o', label='buy-hold drawdown')
plt.title('Drawdown')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()